In [ ]:
"""
╔══════════════════════════════════════════════════════════════╗
║        Project Phantasm Universal — 通用檔案保險箱 GUI        ║
║        phantasm_universal_app.py                            ║
║                                                              ║
║  依賴套件：pip install numpy Pillow                          ║
║  音訊模組使用標準庫 wave（不需額外安裝）                      ║
║                                                              ║
║  加密輸出：stego.wav + key.png + db_value.txt                ║
║  解密輸入：stego.wav + key.png + db_value.txt + 口令          ║
║                                                              ║
║  支援：txt / pdf / docx / png / jpg / mp3 / json / 任意檔案   ║
╚══════════════════════════════════════════════════════════════╝
"""

import os
import wave
import json
import random
import secrets
import hashlib
import threading
import numpy as np
import tkinter as tk
from tkinter import ttk, filedialog, messagebox
from PIL import Image, ImageTk


# ══════════════════════════════════════════════════════════════
#  ███  CORE LOGIC  ███
#  混沌映射 + bytes 打亂（通用檔案加密 / 解密）
# ══════════════════════════════════════════════════════════════

def generate_logistic_sequence(size: int, x0: float, r: float = 3.99) -> np.ndarray:
    """
    Logistic Map：x_{n+1} = r * x_n * (1 - x_n)
    r=3.99 → 混沌區，數列對 x0 的微小變動非常敏感。
    """
    x = x0
    seq = np.zeros(size)
    for i in range(size):
        x = r * x * (1 - x)
        seq[i] = x
    return seq


def encrypt_bytes_to_noise(data_bytes: bytes, x0: float) -> np.ndarray:
    """
    任意檔案 bytes → noise bytes。
    原理：用混沌數列的排序索引對 byte 位置做非線性置換。
    """
    arr = np.frombuffer(data_bytes, dtype=np.uint8).copy()
    chaos_seq = generate_logistic_sequence(len(arr), x0)
    shuffle_indices = np.argsort(chaos_seq)
    encrypted = arr[shuffle_indices]
    return encrypted


def decrypt_noise_to_bytes(noise_array: np.ndarray, x0: float) -> bytes:
    """
    noise bytes → 原始檔案 bytes。
    原理：用相同混沌索引做逆映射。
    """
    flat = noise_array.flatten()
    chaos_seq = generate_logistic_sequence(len(flat), x0)
    shuffle_indices = np.argsort(chaos_seq)

    recovered = np.zeros(len(flat), dtype=np.uint8)
    recovered[shuffle_indices] = flat
    return recovered.tobytes()


# ══════════════════════════════════════════════════════════════
#  ███  FILE METADATA  ███
#  檔名 / 副檔名 / 大小資訊
# ══════════════════════════════════════════════════════════════

def build_metadata(secret_file_path: str) -> bytes:
    """
    建立檔案 metadata，解密時用來還原原始檔名。
    """
    filename = os.path.basename(secret_file_path)
    file_size = os.path.getsize(secret_file_path)

    meta = {
        "filename": filename,
        "file_size": file_size,
        "format": "PHANTASM_UNIVERSAL_V1"
    }
    return json.dumps(meta, ensure_ascii=False).encode("utf-8")


def parse_metadata(meta_bytes: bytes) -> dict:
    """解析 metadata。"""
    return json.loads(meta_bytes.decode("utf-8"))


def safe_output_path(output_dir: str, filename: str) -> str:
    """
    避免解密輸出覆蓋既有檔案。
    """
    base, ext = os.path.splitext(filename)
    candidate = os.path.join(output_dir, filename)
    count = 1
    while os.path.exists(candidate):
        candidate = os.path.join(output_dir, f"{base}_recovered_{count}{ext}")
        count += 1
    return candidate


# ══════════════════════════════════════════════════════════════
#  ███  MEDIA PNG  ███
#  LSB 金鑰隱寫（隱藏 / 提取 x0 浮點數）
# ══════════════════════════════════════════════════════════════

def _get_salt(password: str) -> int:
    """口令 → SHA-256 → 取前 8 hex 位作為整數鹽值。"""
    h = hashlib.sha256(password.encode()).hexdigest()
    return int(h[:8], 16)


def hide_key_to_png(png_path: str, key_float: float, password: str, output_path: str) -> int:
    """
    把混沌金鑰 x0 用 LSB 隱寫進 PNG。
    x0 使用 IEEE 754 double，完整 64-bit 儲存。
    db_value = real_offset XOR SHA256(口令)[:8]
    """
    import struct

    key_bytes = struct.pack(">d", key_float)
    key_bits = "".join(f"{b:08b}" for b in key_bytes)

    img = Image.open(png_path).convert("RGB")
    img_arr = np.array(img, dtype=np.uint8)
    height, width = img_arr.shape[:2]
    total_pixels = width * height

    n_bits = len(key_bits)
    step = 3
    needed = n_bits * step + 1
    max_safe = total_pixels - needed
    if max_safe < 0:
        raise ValueError(
            f"PNG 圖片太小（僅 {total_pixels} 像素），需要至少 {needed} 像素。"
        )

    real_offset = random.randint(0, min(5000, max_safe))
    salt = _get_salt(password)
    db_value = real_offset ^ salt

    for i, bit in enumerate(key_bits):
        idx = real_offset + i * step
        row, col = idx // width, idx % width
        img_arr[row, col, 0] = (img_arr[row, col, 0] & 0xFE) | int(bit)

    Image.fromarray(img_arr).save(output_path)
    return db_value


def extract_key_from_png(png_path: str, db_value: int, password: str) -> float:
    """從 PNG 還原 x0 金鑰。"""
    import struct

    salt = _get_salt(password)
    real_offset = db_value ^ salt

    img = Image.open(png_path).convert("RGB")
    img_arr = np.array(img, dtype=np.uint8)
    height, width = img_arr.shape[:2]
    total_pixels = width * height

    n_bits = 64
    step = 3
    bits = []
    for i in range(n_bits):
        idx = real_offset + i * step
        if idx >= total_pixels:
            raise ValueError(
                "PNG 金鑰提取失敗：像素索引超出圖片範圍。\n"
                "請確認 key.png、db_value.txt 和口令三者來自同一次加密。"
            )
        row, col = idx // width, idx % width
        bits.append(str(img_arr[row, col, 0] & 1))

    bit_str = "".join(bits)
    raw_bytes = bytes(int(bit_str[i:i + 8], 2) for i in range(0, 64, 8))
    return struct.unpack(">d", raw_bytes)[0]


# ══════════════════════════════════════════════════════════════
#  ███  MEDIA WAV  ███
#  LSB 通用 bytes 隱寫進 WAV 音訊
# ══════════════════════════════════════════════════════════════

def hide_bytes_to_wav(wav_path: str, encrypted_bytes: np.ndarray, meta_bytes: bytes, save_path: str) -> tuple:
    """
    把任意加密後 bytes LSB 藏進 WAV 音訊。

    資料格式：
      magic: b"PHANTASM_U1"              11 bytes
      metadata 長度                      4 bytes
      metadata JSON bytes
      encrypted data 長度                8 bytes
      encrypted data bytes
      end marker: b"PHANTASM_END"        12 bytes
    """
    try:
        magic = b"PHANTASM_U1"
        end_marker = b"PHANTASM_END"
        data_bytes = encrypted_bytes.astype(np.uint8).tobytes()

        payload = (
            magic +
            len(meta_bytes).to_bytes(4, "big") +
            meta_bytes +
            len(data_bytes).to_bytes(8, "big") +
            data_bytes +
            end_marker
        )

        binary_msg = "".join(format(b, "08b") for b in payload)

        with wave.open(wav_path, "rb") as audio:
            params = audio.getparams()
            frame_bytes = bytearray(audio.readframes(audio.getnframes()))

        if len(binary_msg) > len(frame_bytes):
            needed_kb = len(binary_msg) / 8 / 1024
            avail_kb = len(frame_bytes) / 1024
            return False, (
                f"WAV 容量不足！\n"
                f"需要約 {needed_kb:.1f} KB，WAV 僅有 {avail_kb:.1f} KB。\n"
                f"請換更長的 WAV，或縮小要隱藏的檔案。"
            )

        for i, bit in enumerate(binary_msg):
            frame_bytes[i] = (frame_bytes[i] & 0xFE) | int(bit)

        with wave.open(save_path, "wb") as f:
            f.setparams(params)
            f.writeframes(bytes(frame_bytes))

        return True, save_path
    except Exception as e:
        return False, str(e)


def extract_bytes_from_wav(wav_path: str) -> tuple:
    """
    從 WAV 提取 metadata 與加密資料。
    回傳：(encrypted_array, metadata_dict) 或 (None, None)
    """
    try:
        with wave.open(wav_path, "rb") as audio:
            frame_bytes = bytearray(audio.readframes(audio.getnframes()))

        def get_byte(start_bit: int) -> int:
            bits = "".join(str(frame_bytes[j] & 1) for j in range(start_bit, start_bit + 8))
            return int(bits, 2)

        cursor = 0

        def read_bytes(n: int) -> bytes:
            nonlocal cursor
            out = bytes(get_byte(i) for i in range(cursor, cursor + n * 8, 8))
            cursor += n * 8
            return out

        magic = read_bytes(len(b"PHANTASM_U1"))
        if magic != b"PHANTASM_U1":
            raise ValueError("這不是 Phantasm Universal 產生的 WAV，或檔案已損壞。")

        meta_len = int.from_bytes(read_bytes(4), "big")
        meta_bytes = read_bytes(meta_len)
        metadata = parse_metadata(meta_bytes)

        data_len = int.from_bytes(read_bytes(8), "big")
        encrypted_data = read_bytes(data_len)

        end_marker = read_bytes(len(b"PHANTASM_END"))
        if end_marker != b"PHANTASM_END":
            raise ValueError("結束標記不正確，可能資料被截斷或 WAV 已損壞。")

        arr = np.frombuffer(encrypted_data, dtype=np.uint8)
        return arr, metadata
    except Exception:
        return None, None


# ══════════════════════════════════════════════════════════════
#  ███  KEY FILE  ███
#  x0 自動產生 + db_value txt 存取
# ══════════════════════════════════════════════════════════════

def generate_x0() -> float:
    """用密碼學安全亂數產生 x0。"""
    while True:
        x0 = secrets.randbelow(10**15) / 10**15
        if 0.02 < x0 < 0.98:
            return x0


def save_db_value(db_val: int, x0: float, txt_path: str):
    """將 db_value 寫成 txt。"""
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write("# Project Phantasm Universal — 解密金鑰檔案\n")
        f.write("# 解密時需要：stego.wav + key.png + db_value.txt + 口令\n\n")
        f.write(f"db_value={db_val}\n")
        f.write("# x0_ref 僅供除錯參考，真正解密仍會從 PNG 提取 x0\n")
        f.write(f"x0_ref={x0:.15f}\n")


def load_db_value_from_txt(txt_path: str) -> int:
    """從 txt 讀回 db_value。"""
    with open(txt_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line.startswith("db_value="):
                return int(line.split("=", 1)[1])
    raise ValueError("找不到 db_value 欄位，請確認是 Phantasm Universal 產生的 txt 檔案。")


# ══════════════════════════════════════════════════════════════
#  ███  INTEGRATOR  ███
#  完整加密 / 解密流程
# ══════════════════════════════════════════════════════════════

def full_encrypt(secret_file: str, cover_wav: str, key_png: str,
                 password: str, out_wav: str, out_key_png: str, out_txt: str,
                 progress_cb=None) -> tuple:
    """
    完整加密流程：
      1. 讀取任意檔案 bytes
      2. 產生 x0
      3. bytes 混沌置換
      4. 加密 bytes 藏進 WAV
      5. x0 藏進 PNG，db_value 存 txt
    """
    try:
        def step(msg):
            if progress_cb:
                progress_cb(msg)

        step("【準備】讀取秘密檔案 bytes...")
        with open(secret_file, "rb") as f:
            data_bytes = f.read()
        meta_bytes = build_metadata(secret_file)
        step(f"  檔案：{os.path.basename(secret_file)}")
        step(f"  大小：{len(data_bytes):,} bytes")

        step("【步驟 1/4】自動產生密碼學安全金鑰 x0...")
        x0 = generate_x0()
        step("  x0 已產生，每次加密唯一")

        step("【步驟 2/4】混沌映射打亂檔案 bytes...")
        encrypted_arr = encrypt_bytes_to_noise(data_bytes, x0)
        step(f"  已打亂 {len(encrypted_arr):,} bytes")

        step("【步驟 3/4】將加密 bytes LSB 隱寫進 WAV...")
        ok, result = hide_bytes_to_wav(cover_wav, encrypted_arr, meta_bytes, out_wav)
        if not ok:
            return False, result, 0.0
        step(f"  輸出：{os.path.basename(out_wav)}")

        step("【步驟 4/4】將 x0 LSB 隱寫進 PNG，db_value 存成 txt...")
        db_val = hide_key_to_png(key_png, x0, password, out_key_png)
        save_db_value(db_val, x0, out_txt)
        step(f"  輸出：{os.path.basename(out_key_png)}")
        step(f"  db_value 已存入：{os.path.basename(out_txt)}")

        step("\n✅ 加密完成！")
        return True, str(db_val), x0
    except Exception as e:
        return False, str(e), 0.0


def full_decrypt(stego_wav: str, key_png: str, password: str,
                 db_value: int, output_dir: str, progress_cb=None) -> tuple:
    """
    完整解密流程：
      1. 從 PNG + db_value + 口令還原 x0
      2. 從 WAV 提取 encrypted bytes + metadata
      3. 混沌逆映射還原原檔 bytes
      4. 按原檔名輸出
    """
    try:
        def step(msg):
            if progress_cb:
                progress_cb(msg)

        step("【步驟 1/3】從 PNG 提取混沌金鑰 x0...")
        x0 = extract_key_from_png(key_png, db_value, password)
        if not (0.0 < x0 < 1.0):
            return False, "x0 還原值異常，請確認 key.png、db_value.txt 和口令正確。"
        step(f"  x0 已還原，值域驗證通過")

        step("【步驟 2/3】從 WAV 提取加密 bytes 與 metadata...")
        encrypted_arr, metadata = extract_bytes_from_wav(stego_wav)
        if encrypted_arr is None:
            return False, "從 WAV 提取資料失敗，可能 WAV 不是本程式產生或檔案已損壞。"
        step(f"  原始檔名：{metadata.get('filename', 'unknown.bin')}")
        step(f"  加密資料大小：{len(encrypted_arr):,} bytes")

        step("【步驟 3/3】混沌逆映射還原原始檔案...")
        recovered_bytes = decrypt_noise_to_bytes(encrypted_arr, x0)

        expected_size = metadata.get("file_size")
        if expected_size is not None and len(recovered_bytes) != expected_size:
            return False, "還原後檔案大小不符合 metadata，可能口令或金鑰檔案錯誤。"

        filename = metadata.get("filename", "recovered_file.bin")
        out_path = safe_output_path(output_dir, filename)
        with open(out_path, "wb") as f:
            f.write(recovered_bytes)

        step(f"  輸出：{out_path}")
        step("\n✅ 解密完成！")
        return True, out_path
    except Exception as e:
        return False, str(e)


# ══════════════════════════════════════════════════════════════
#  ███  GUI  ███
# ══════════════════════════════════════════════════════════════

C = {
    "bg": "#0d1117",
    "surface": "#161b22",
    "border": "#30363d",
    "accent": "#58a6ff",
    "accent2": "#3fb950",
    "accent3": "#f85149",
    "accent4": "#d2a8ff",
    "text": "#e6edf3",
    "subtext": "#8b949e",
    "entry_bg": "#21262d",
}

FONT_TITLE = ("Consolas", 22, "bold")
FONT_HEAD = ("Consolas", 13, "bold")
FONT_BODY = ("Microsoft JhengHei", 11)
FONT_MONO = ("Consolas", 10)
FONT_STATUS = ("Consolas", 10)


class PhantasmUniversalApp(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Project Phantasm Universal — 通用檔案保險箱")
        self.geometry("980x720")
        self.minsize(880, 640)
        self.configure(bg=C["bg"])
        self.resizable(True, True)

        self.status_var = tk.StringVar(value="就緒")
        self.db_value = tk.StringVar(value="")

        self._build_title_bar()
        self._build_notebook()
        self._build_status_bar()

    def _build_title_bar(self):
        bar = tk.Frame(self, bg=C["surface"], height=56)
        bar.pack(fill="x")
        bar.pack_propagate(False)
        tk.Label(bar, text="⬡  Project Phantasm Universal",
                 font=FONT_TITLE, fg=C["accent"], bg=C["surface"]).pack(side="left", padx=20, pady=8)
        tk.Label(bar, text="任意檔案 bytes 隱寫安全系統",
                 font=FONT_BODY, fg=C["subtext"], bg=C["surface"]).pack(side="left", padx=4)
        tk.Frame(self, bg=C["border"], height=1).pack(fill="x")

    def _build_notebook(self):
        style = ttk.Style(self)
        style.theme_use("clam")
        style.configure("P.TNotebook", background=C["bg"], borderwidth=0, tabmargins=[0, 4, 0, 0])
        style.configure("P.TNotebook.Tab", background=C["surface"], foreground=C["subtext"],
                        font=FONT_HEAD, padding=[18, 8], borderwidth=0)
        style.map("P.TNotebook.Tab", background=[("selected", C["bg"])], foreground=[("selected", C["accent"])])

        nb = ttk.Notebook(self, style="P.TNotebook")
        nb.pack(fill="both", expand=True)

        enc = tk.Frame(nb, bg=C["bg"])
        dec = tk.Frame(nb, bg=C["bg"])
        abt = tk.Frame(nb, bg=C["bg"])
        nb.add(enc, text="  🔒  加密  ")
        nb.add(dec, text="  🔓  解密  ")
        nb.add(abt, text="  ℹ  關於  ")

        self._build_encrypt_page(enc)
        self._build_decrypt_page(dec)
        self._build_about_page(abt)

    def _build_status_bar(self):
        tk.Frame(self, bg=C["border"], height=1).pack(fill="x")
        bar = tk.Frame(self, bg=C["surface"], height=28)
        bar.pack(fill="x", side="bottom")
        bar.pack_propagate(False)
        tk.Label(bar, text="●", fg=C["accent2"], bg=C["surface"], font=("Consolas", 10)).pack(side="left", padx=8)
        tk.Label(bar, textvariable=self.status_var, fg=C["subtext"], bg=C["surface"],
                 font=FONT_STATUS, anchor="w").pack(side="left")

    def set_status(self, msg: str):
        self.status_var.set(msg)
        self.update_idletasks()

    def _build_encrypt_page(self, parent):
        left = tk.Frame(parent, bg=C["bg"])
        right = tk.Frame(parent, bg=C["bg"])
        left.pack(side="left", fill="both", expand=True, padx=(20, 8), pady=20)
        right.pack(side="right", fill="both", expand=True, padx=(8, 20), pady=20)

        self._section(left, "輸入設定", C["accent"])
        self.enc_secret = self._file_row(left, "📦  要隱藏的任意檔案", "選擇檔案", [("所有檔案", "*.*")])
        self.enc_wav = self._file_row(left, "🎵  封面 WAV 音訊（藏檔案 bytes 用）", "選擇 WAV", [("WAV 音訊", "*.wav")])
        self.enc_keypng = self._file_row(left, "🔑  金鑰載體 PNG（藏 x0 用）", "選擇 PNG", [("PNG 圖片", "*.png")])

        self._label(left, "保護口令")
        self.enc_pw = self._entry(left, "在此輸入口令", show="●")

        self._section(left, "輸出設定（三個檔案）", C["accent"])
        self.enc_out_wav = self._save_row(left, "💾  輸出 WAV", "stego_output.wav", [("WAV 音訊", "*.wav")])
        self.enc_out_png = self._save_row(left, "💾  輸出金鑰 PNG", "key_output.png", [("PNG 圖片", "*.png")])
        self.enc_out_txt = self._save_row(left, "💾  db_value.txt", "db_value.txt", [("文字檔案", "*.txt")])

        self._section(right, "執行加密", C["accent"])
        hint = tk.Frame(right, bg=C["surface"], highlightbackground=C["border"], highlightthickness=1)
        hint.pack(fill="x", pady=(0, 8))
        tk.Label(hint,
                 text="📦 支援任意檔案：txt / pdf / docx / png / jpg / json / mp3 / zip... 都會以 bytes 形式處理",
                 fg=C["accent2"], bg=C["surface"], font=FONT_MONO, wraplength=430,
                 justify="left").pack(anchor="w", padx=10, pady=8)

        self.enc_log = self._log_box(right)

        dv_frame = tk.Frame(right, bg=C["surface"], highlightbackground=C["border"], highlightthickness=1)
        dv_frame.pack(fill="x", pady=(8, 0))
        tk.Label(dv_frame, text="🗝  db_value（已自動存入 txt）", fg=C["accent"],
                 bg=C["surface"], font=FONT_MONO).pack(anchor="w", padx=10, pady=(8, 2))
        dv_inner = tk.Frame(dv_frame, bg=C["surface"])
        dv_inner.pack(fill="x", padx=10, pady=(0, 8))
        self.dv_display = tk.Entry(dv_inner, textvariable=self.db_value, state="readonly",
                                   font=FONT_MONO, bg=C["entry_bg"], fg=C["accent2"],
                                   readonlybackground=C["entry_bg"], relief="flat", bd=0)
        self.dv_display.pack(side="left", fill="x", expand=True, ipady=5, padx=(0, 8))
        tk.Button(dv_inner, text="複製", font=FONT_MONO, bg=C["border"], fg=C["text"],
                  relief="flat", activebackground=C["accent"], activeforeground="white",
                  cursor="hand2", command=self._copy_dbval).pack(side="right")

        self._big_button(right, "▶  開始加密", C["accent"], self._run_encrypt)

    def _build_decrypt_page(self, parent):
        left = tk.Frame(parent, bg=C["bg"])
        right = tk.Frame(parent, bg=C["bg"])
        left.pack(side="left", fill="both", expand=True, padx=(20, 8), pady=20)
        right.pack(side="right", fill="both", expand=True, padx=(8, 20), pady=20)

        self._section(left, "輸入設定", C["accent4"])
        self.dec_wav = self._file_row(left, "🎵  含密文的 WAV", "選擇 WAV", [("WAV 音訊", "*.wav")])
        self.dec_keypng = self._file_row(left, "🔑  含金鑰的 PNG", "選擇 PNG", [("PNG 圖片", "*.png")])

        self._label(left, "📄  db_value.txt → 選擇後自動填入 db_value")
        dbtxt_row = tk.Frame(left, bg=C["bg"])
        dbtxt_row.pack(fill="x", pady=(0, 2))
        self.dec_txt_var = tk.StringVar()
        tk.Entry(dbtxt_row, textvariable=self.dec_txt_var, font=FONT_MONO,
                 bg=C["entry_bg"], fg=C["text"], insertbackground=C["text"],
                 relief="flat", bd=0).pack(side="left", fill="x", expand=True, ipady=6, ipadx=8)
        tk.Button(dbtxt_row, text="選擇 txt", font=FONT_MONO, bg=C["border"], fg=C["text"],
                  relief="flat", activebackground=C["accent4"], activeforeground="white",
                  cursor="hand2", padx=10, command=self._load_txt_file).pack(side="right", padx=(6, 0))
        tk.Frame(left, bg=C["border"], height=1).pack(fill="x")

        self._label(left, "db_value 數字")
        self.dec_dbval = self._entry(left, "例如：2857392")

        self._label(left, "口令")
        self.dec_pw = self._entry(left, "在此輸入口令", show="●")

        self._section(left, "輸出設定", C["accent4"])
        self.dec_out_dir = self._dir_row(left, "📁  還原檔案輸出資料夾")

        self._section(right, "執行解密", C["accent4"])
        self.dec_log = self._log_box(right)

        info_frame = tk.Frame(right, bg=C["surface"], highlightbackground=C["border"], highlightthickness=1)
        info_frame.pack(fill="both", expand=True, pady=(8, 0))
        tk.Label(info_frame, text="📦  還原資訊", fg=C["accent4"], bg=C["surface"],
                 font=FONT_MONO).pack(anchor="w", padx=10, pady=(8, 4))
        self.recovered_info = tk.Label(info_frame, bg=C["entry_bg"], text="解密後會顯示輸出檔案路徑",
                                       fg=C["subtext"], font=FONT_BODY, wraplength=430, justify="left")
        self.recovered_info.pack(fill="both", expand=True, padx=10, pady=(0, 10))

        self._big_button(right, "▶  開始解密", C["accent4"], self._run_decrypt)

    def _build_about_page(self, parent):
        canvas = tk.Canvas(parent, bg=C["bg"], highlightthickness=0)
        sb = ttk.Scrollbar(parent, orient="vertical", command=canvas.yview)
        canvas.configure(yscrollcommand=sb.set)
        sb.pack(side="right", fill="y")
        canvas.pack(side="left", fill="both", expand=True)
        frame = tk.Frame(canvas, bg=C["bg"])
        canvas.create_window((0, 0), window=frame, anchor="nw")
        frame.bind("<Configure>", lambda e: canvas.configure(scrollregion=canvas.bbox("all")))

        def lbl(text, font, fg, pady=(2, 2), padx=0):
            tk.Label(frame, text=text, font=font, fg=fg, bg=C["bg"],
                     justify="left", wraplength=860).pack(anchor="w", pady=pady, padx=(40 + padx, 40))

        lbl("Project Phantasm Universal", ("Consolas", 26, "bold"), C["accent"], (20, 2))
        lbl("通用檔案版本：任何檔案最終都可以視為 bytes，因此可以被統一加密與隱寫。",
            FONT_BODY, C["subtext"], (2, 16))
        lbl("🔒 加密流程", FONT_HEAD, C["accent2"], (14, 4))
        lbl("1. 讀取任意檔案 bytes\n2. 用混沌映射打亂 byte 順序\n3. 把加密後 bytes 藏入 WAV LSB\n4. 把 x0 金鑰藏入 PNG LSB\n5. db_value 存入 txt", FONT_BODY, C["text"], (0, 10), 8)
        lbl("🔓 解密流程", FONT_HEAD, C["accent4"], (14, 4))
        lbl("1. 使用 key.png + db_value.txt + 口令還原 x0\n2. 從 WAV 提取加密 bytes 和 metadata\n3. 用 x0 做逆映射\n4. 依 metadata 還原原始檔名與內容", FONT_BODY, C["text"], (0, 10), 8)
        lbl("⚠ 注意", FONT_HEAD, C["accent3"], (14, 4))
        lbl("這是專題展示級安全設計，不建議直接拿來保護真正高敏感資料。真正正式用途應加入 AES-GCM、HMAC、KDF、檔案完整性驗證等機制。", FONT_BODY, C["text"], (0, 10), 8)

    def _run_encrypt(self):
        secret = self.enc_secret.get().strip()
        wav = self.enc_wav.get().strip()
        keypng = self.enc_keypng.get().strip()
        pw = self.enc_pw.get().strip()
        o_wav = self.enc_out_wav.get().strip()
        o_png = self.enc_out_png.get().strip()
        o_txt = self.enc_out_txt.get().strip()

        errors = []
        if not os.path.exists(secret): errors.append("找不到要隱藏的檔案")
        if not os.path.exists(wav): errors.append("找不到封面 WAV 音訊")
        if not os.path.exists(keypng): errors.append("找不到金鑰載體 PNG")
        if not pw: errors.append("口令不能為空")
        if not o_wav: errors.append("請設定輸出 WAV 路徑")
        if not o_png: errors.append("請設定輸出金鑰 PNG 路徑")
        if not o_txt: errors.append("請設定 db_value txt 路徑")

        if errors:
            messagebox.showerror("輸入錯誤", "\n".join(f"• {e}" for e in errors))
            return

        self._log_clear(self.enc_log)
        self.db_value.set("")

        def task():
            ok, result, x0 = full_encrypt(secret, wav, keypng, pw, o_wav, o_png, o_txt,
                                          progress_cb=lambda m: self._log_append(self.enc_log, m))
            if ok:
                self.db_value.set(result)
                self._log_append(self.enc_log,
                                 f"\n🗝  db_value = {result}"
                                 f"\n📄  已自動存入：{o_txt}"
                                 f"\n⚠  解密需要 stego.wav + key.png + db_value.txt + 口令")
                self.set_status("加密完成")
            else:
                self._log_append(self.enc_log, f"\n❌ 加密失敗：{result}")
                self.set_status("加密失敗")

        threading.Thread(target=task, daemon=True).start()

    def _run_decrypt(self):
        wav = self.dec_wav.get().strip()
        keypng = self.dec_keypng.get().strip()
        pw = self.dec_pw.get().strip()
        dbv = self.dec_dbval.get().strip()
        out_dir = self.dec_out_dir.get().strip()

        errors = []
        if not os.path.exists(wav): errors.append("找不到 WAV 音訊")
        if not os.path.exists(keypng): errors.append("找不到金鑰 PNG")
        if not pw: errors.append("口令不能為空")
        if not os.path.isdir(out_dir): errors.append("請設定有效的輸出資料夾")
        try:
            db_int = int(dbv)
        except ValueError:
            errors.append("db_value 必須是整數")
            db_int = 0

        if errors:
            messagebox.showerror("輸入錯誤", "\n".join(f"• {e}" for e in errors))
            return

        self._log_clear(self.dec_log)
        self.recovered_info.configure(text="解密中...")

        def task():
            ok, result = full_decrypt(wav, keypng, pw, db_int, out_dir,
                                      progress_cb=lambda m: self._log_append(self.dec_log, m))
            if ok:
                self.set_status("解密完成")
                self.recovered_info.configure(text=f"✅ 已還原檔案：\n{result}")
            else:
                self._log_append(self.dec_log, f"\n❌ 解密失敗：{result}")
                self.recovered_info.configure(text="解密失敗")
                self.set_status("解密失敗")

        threading.Thread(target=task, daemon=True).start()

    def _load_txt_file(self):
        path = filedialog.askopenfilename(title="選擇 db_value.txt", filetypes=[("文字檔案", "*.txt"), ("所有檔案", "*.*")])
        if not path:
            return
        try:
            db_val = load_db_value_from_txt(path)
            self.dec_txt_var.set(path)
            self.dec_dbval.delete(0, "end")
            self.dec_dbval.insert(0, str(db_val))
            self.set_status(f"✓ 已從 txt 讀入 db_value = {db_val}")
        except Exception as e:
            messagebox.showerror("讀取失敗", str(e))

    def _copy_dbval(self):
        val = self.db_value.get()
        if val:
            self.clipboard_clear()
            self.clipboard_append(val)
            self.set_status(f"已複製 db_value：{val}")

    def _section(self, parent, title, color):
        f = tk.Frame(parent, bg=C["bg"])
        f.pack(fill="x", pady=(12, 4))
        tk.Label(f, text=title, font=FONT_HEAD, fg=color, bg=C["bg"]).pack(side="left")
        tk.Frame(f, bg=C["border"], height=1).pack(side="left", fill="x", expand=True, padx=(8, 0), pady=6)

    def _label(self, parent, text):
        tk.Label(parent, text=text, font=FONT_BODY, fg=C["subtext"], bg=C["bg"]).pack(anchor="w", pady=(6, 1))

    def _entry(self, parent, placeholder, show=""):
        e = tk.Entry(parent, font=FONT_MONO, bg=C["entry_bg"], fg=C["text"],
                     insertbackground=C["text"], relief="flat", bd=0, show=show)
        e.insert(0, placeholder)
        e.pack(fill="x", ipady=6, ipadx=8, pady=(0, 2))
        tk.Frame(parent, bg=C["border"], height=1).pack(fill="x")
        return e

    def _file_row(self, parent, label, btn_text, ftypes):
        self._label(parent, label)
        var = tk.StringVar()
        row = tk.Frame(parent, bg=C["bg"])
        row.pack(fill="x", pady=(0, 2))
        tk.Entry(row, textvariable=var, font=FONT_MONO, bg=C["entry_bg"], fg=C["text"],
                 insertbackground=C["text"], relief="flat", bd=0).pack(side="left", fill="x", expand=True, ipady=6, ipadx=8)
        tk.Button(row, text=btn_text, font=FONT_MONO, bg=C["border"], fg=C["text"],
                  relief="flat", bd=0, activebackground=C["accent"], activeforeground="white",
                  cursor="hand2", padx=10,
                  command=lambda: var.set(filedialog.askopenfilename(filetypes=ftypes) or var.get())
                  ).pack(side="right", padx=(6, 0))
        tk.Frame(parent, bg=C["border"], height=1).pack(fill="x")
        return var

    def _save_row(self, parent, label, default, ftypes):
        self._label(parent, label)
        var = tk.StringVar(value=default)
        row = tk.Frame(parent, bg=C["bg"])
        row.pack(fill="x", pady=(0, 2))
        tk.Entry(row, textvariable=var, font=FONT_MONO, bg=C["entry_bg"], fg=C["text"],
                 insertbackground=C["text"], relief="flat", bd=0).pack(side="left", fill="x", expand=True, ipady=6, ipadx=8)
        tk.Button(row, text="瀏覽", font=FONT_MONO, bg=C["border"], fg=C["text"],
                  relief="flat", bd=0, activebackground=C["accent2"], activeforeground="white",
                  cursor="hand2", padx=10,
                  command=lambda: var.set(filedialog.asksaveasfilename(
                      defaultextension=ftypes[0][1].lstrip("*"), filetypes=ftypes) or var.get())
                  ).pack(side="right", padx=(6, 0))
        tk.Frame(parent, bg=C["border"], height=1).pack(fill="x")
        return var

    def _dir_row(self, parent, label):
        self._label(parent, label)
        var = tk.StringVar(value=os.getcwd())
        row = tk.Frame(parent, bg=C["bg"])
        row.pack(fill="x", pady=(0, 2))
        tk.Entry(row, textvariable=var, font=FONT_MONO, bg=C["entry_bg"], fg=C["text"],
                 insertbackground=C["text"], relief="flat", bd=0).pack(side="left", fill="x", expand=True, ipady=6, ipadx=8)
        tk.Button(row, text="選擇資料夾", font=FONT_MONO, bg=C["border"], fg=C["text"],
                  relief="flat", bd=0, activebackground=C["accent4"], activeforeground="white",
                  cursor="hand2", padx=10,
                  command=lambda: var.set(filedialog.askdirectory() or var.get())
                  ).pack(side="right", padx=(6, 0))
        tk.Frame(parent, bg=C["border"], height=1).pack(fill="x")
        return var

    def _log_box(self, parent):
        frame = tk.Frame(parent, bg=C["surface"], highlightbackground=C["border"], highlightthickness=1)
        frame.pack(fill="both", expand=True, pady=(4, 8))
        t = tk.Text(frame, font=FONT_MONO, bg=C["entry_bg"], fg=C["text"],
                    relief="flat", bd=0, wrap="word", height=8,
                    insertbackground=C["text"], state="disabled")
        sb = ttk.Scrollbar(frame, command=t.yview)
        t.configure(yscrollcommand=sb.set)
        sb.pack(side="right", fill="y")
        t.pack(fill="both", expand=True, padx=1, pady=1)
        return t

    def _log_append(self, box, msg):
        box.configure(state="normal")
        box.insert("end", msg + "\n")
        box.see("end")
        box.configure(state="disabled")
        self.set_status(msg.strip())

    def _log_clear(self, box):
        box.configure(state="normal")
        box.delete("1.0", "end")
        box.configure(state="disabled")

    def _big_button(self, parent, text, color, cmd):
        tk.Button(parent, text=text, font=("Consolas", 12, "bold"),
                  bg=color, fg="white", relief="flat", bd=0,
                  activebackground=C["surface"], activeforeground=color,
                  cursor="hand2", pady=10, command=cmd).pack(fill="x", pady=(8, 4))


# ══════════════════════════════════════════════════════════════
if __name__ == "__main__":
    app = PhantasmUniversalApp()
    app.mainloop()
